# MeanShift, CamShift, Optical Flow, Filtr Kalmana

Tematem zajęć są modele pozwalające na wyznaczenie lub predykcję ruchu obiektu na podstawie obserwacji.

| Metoda | Co dostaje na wejściu | Co zwraca |
|---|---|---|
| MeanShift | mapę „jak bardzo piksel pasuje do obiektu” i okno z poprzedniej klatki | nowe położenie okna |
| CamShift | to samo | nowe położenie, rozmiar i obrót okna |
| Przepływ optyczny | dwie kolejne klatki | wektory przesunięcia punktów (wybranych lub wszystkich) |
| Filtr Kalmana | zaszumione pomiary położenia i model ruchu | wygładzone położenie, prędkość i przewidywanie następnego położenia |

Uwaga do przykładowych skryptów wyświetlających obraz (wszystkie poza filtrem Kalmana): funkcja `cv.imshow` otwiera okno graficzne, więc nie działa w Google Colab. W Colab obraz można wyświetlić przez `matplotlib` albo funkcją `cv2_imshow(obraz)` z modułu `google.colab.patches` (przyjmuje sam obraz, bez nazwy okna; wywołana w pętli dokleja kolejne obrazy pod komórką, więc lepiej wyświetlać co którąś klatkę).

## MeanShift

MeanShift to metoda nieparametryczna oparta na estymacji funkcji gęstości. Wykorzystując gradient rozkładu prawdopodobieństwa, algorytm ten odnajduje najbliższe maksimum lokalne i umożliwia skuteczne śledzenie obiektów na podstawie rozkładu prawdopodobieństwa ich kolorów.

Mówiąc prościej: dla każdego piksela wyznaczamy liczbę mówiącą, jak bardzo jego kolor pasuje do śledzonego obiektu. Powstaje mapa, na której obiekt jest jasną plamą. Okno śledzenia przesuwamy krok po kroku tam, gdzie w jego obrębie skupia się najwięcej „jasności”, aż przestanie się ruszać.

### Algorytm

**Inicjalizacja (wykonywana raz, na pierwszej klatce).** Wybierane jest okno startowe $W$ o rozmiarze $h \times w$, obejmujące obiekt. W oknie tym obliczany jest histogram obiektu $H(c)$, gdzie $c$ oznacza kolor (np. składową H, czyli barwę, w przestrzeni HSV). Histogram mówi, jak często dany kolor występuje w obiekcie. Jest on modelem obiektu i w dalszych krokach się nie zmienia.

**Dla każdej kolejnej klatki:**

1. Dla każdego piksela klatki tworzona jest mapa prawdopodobieństwa:

$$P(x,y) = H(c(x,y))$$

Gdzie $c(x,y)$ - kolor piksela o współrzędnych $(x,y)$, $H(c(x,y))$ - wartość histogramu dla koloru, który jest w punkcie $(x,y)$. $P$ nazywa się też obrazem projekcji wstecznej (back projection image). Każdy piksel zostaje więc zastąpiony odpowiedzią na pytanie „jak często twój kolor występował w obiekcie?”.

Okno śledzenia $W_0$ ma środek $(x_0, y_0)$ w miejscu, w którym algorytm zakończył pracę na poprzedniej klatce (dla pierwszej śledzonej klatki jest to środek okna startowego). Indeks $k = 0, 1, 2, \dots$ numeruje kolejne przesunięcia okna w obrębie tej samej klatki: $W_k$ to okno po $k$ przesunięciach, a $(x_k, y_k)$ to jego środek. Zaczynamy od $k=0$.

2. W aktualnym oknie śledzenia $W_k$ obliczany jest środek masy rozkładu prawdopodobieństwa:

$$(x_{c},y_{c}) = (\frac{M_{10}}{M_{00}}, \frac{M_{01}}{M_{00}})$$

Gdzie zerowy moment:
$$M_{00}=\sum_{(x,y)\in W_k}P(x,y)$$
momenty pierwszego rzędu:
$$M_{10}=\sum_{(x,y)\in W_k}xP(x,y)$$
$$M_{01}=\sum_{(x,y)\in W_k}yP(x,y)$$

3. Aktualizujemy położenie okna, przenosząc jego środek do środka masy:

$$(x_{k+1}, y_{k+1}) = (x_{k}, y_{k}) + (\Delta x, \Delta y) = (x_{k}, y_{k}) + (x_{c} - x_{k}, y_{c} - y_{k}) = (x_c, y_c)$$

Kroki 2 i 3 powtarzamy (zwiększając $k$) tak długo, aż prawdziwy będzie warunek:

$$\sqrt{\Delta x^2 + \Delta y^2} < \epsilon$$

Gdzie $\epsilon$ jest pewną niedużą wartością, innymi słowy przesuwamy okno tak długo, aż rozwiązanie się ustabilizuje. W praktyce dodaje się też limit liczby iteracji (w przykładowym skrypcie: najwyżej 10 iteracji, $\epsilon = 1$ piksel).

Wynikiem algorytmu MeanShift jest środek okna $(x_{k+1}, y_{k+1})$ dla pierwszego $k$ spełniającego warunek z $\epsilon$. Rozmiar okna się nie zmienia.

### Jak czytać momenty

Mapę $P$ w oknie można traktować jak histogram położeń: wartość $P(x,y)$ to „liczba głosów” oddanych na punkt $(x,y)$.

- $M_{00}$ to suma wszystkich głosów w oknie, czyli „ile obiektu” widzi okno. Jeśli $M_{00} = 0$, w oknie nie ma ani jednego pasującego piksela i środka masy nie da się policzyć.
- $M_{10}/M_{00}$ i $M_{01}/M_{00}$ to średnie ważone współrzędnych, czyli punkt, wokół którego skupiają się głosy. Same $M_{10}$ i $M_{01}$ nie są położeniem; sens ma dopiero iloraz.

### Przykład (jeden wymiar)

Mapa $P$ ma wartość 1 dla $x = 15, \dots, 19$ (obiekt) i 0 poza tym. Okno obejmuje środek ± 5 pikseli i startuje w $x_0 = 12$.

| Iteracja | Okno | Piksele obiektu w oknie | Środek masy $x_c$ | Przesunięcie |
|---|---|---|---|---|
| 1 | 7–17 | 15, 16, 17 | 16 | 4 |
| 2 | 11–21 | 15–19 | 17 | 1 |
| 3 | 12–22 | 15–19 | 17 | 0, koniec |

W pierwszej iteracji okno tylko zahacza o obiekt, a mimo to po dwóch przesunięciach stoi dokładnie na jego środku (trzecia iteracja tylko to potwierdza).

### O czym warto pamiętać

- **Szukanie jest lokalne.** Okno wspina się po mapie z miejsca, w którym stoi. Jeśli w starym położeniu nie obejmuje żadnego piksela obiektu, nie ma dokąd się przesunąć.
- **Tło o podobnym kolorze dostaje te same wartości $P$ co obiekt**, więc okno może na nie „spłynąć”.
- **Algorytm nie zgłasza zgubienia obiektu.** Zawsze zwraca jakieś okno.
- **$P$ nie musi być prawdopodobieństwem w ścisłym sensie.** W przykładowym skrypcie histogram jest liczony ze składowej H (z pominięciem pikseli o małym nasyceniu lub małej jasności, dla których barwa jest przypadkowa) i skalowany do zakresu 0–255. Dla algorytmu nie ma to znaczenia, bo środek masy jest ilorazem i nie zależy od skali $P$.

Przykładowa implementacja w Pythonie [meanshift](src/meanshift.py). W skrypcie histogram (`cv.calcHist`) jest liczony przed pętlą, a w pętli po klatkach są tylko projekcja wsteczna (`cv.calcBackProject`) i `cv.meanShift`.

## CamShift

CamShift (Continuously Adaptive Mean Shift) jest rozszerzeniem algorytmu MeanShift. Służy do śledzenia obiektów na podstawie rozkładu prawdopodobieństwa, którym często jest histogram koloru. W odróżnieniu od MeanShift dopasowuje do obiektu także rozmiar i orientację okna, co jest potrzebne, gdy obiekt zbliża się, oddala lub obraca.

1. Wykonaj algorytm MeanShift, zwróć nowy środek okna $(x_{c},y_{c})$.

2. Obliczamy nową wielkość i orientację (kąt obrotu) $\theta$ okna.

Potrzebne są do tego momenty centralne drugiego rzędu, liczone w oknie $W$ względem środka masy:

$$\mu_{20}=\sum_{(x,y)\in W}(x-x_{c})^2P(x,y)$$

$$\mu_{02}=\sum_{(x,y)\in W}(y-y_{c})^2P(x,y)$$

$$\mu_{11}=\sum_{(x,y)\in W}(x-x_{c})(y-y_{c})P(x,y)$$

Po podzieleniu przez $M_{00}$ tworzą one macierz kowariancji $C$ położeń pikseli obiektu:

$$C=\frac{1}{M_{00}}\begin{bmatrix}
  \mu_{20}&\mu_{11} \\
  \mu_{11}&\mu_{02}
\end{bmatrix}$$

Kąt rotacji okna (kierunek, wzdłuż którego obiekt jest najbardziej rozciągnięty):

$$\theta=\frac{1}{2}\,\mathrm{atan2}(2\mu_{11},\ \mu_{20}-\mu_{02})$$

Funkcja atan2 to dwuargumentowy arcus tangens (`np.arctan2`), który uwzględnia znaki obu argumentów. Kąt jest mierzony od osi $x$ w stronę osi $y$. Oś $y$ obrazu jest skierowana w dół, więc na ekranie dodatni kąt oznacza obrót zgodny z ruchem wskazówek zegara.

Długości boków są równe przeskalowanym pierwiastkom z wartości własnych macierzy kowariancji $C$:

$$l=k\sqrt{\lambda_{1}}, \qquad s=k\sqrt{\lambda_{2}}$$

Gdzie $l$ to długość (bok wzdłuż kierunku $\theta$), $s$ to szerokość, $k$ to współczynnik skali, a $\lambda_{1} \ge \lambda_{2}$ to wartości własne macierzy $C$. Oznaczając elementy macierzy $C$ jako $a = \mu_{20}/M_{00}$, $b = \mu_{11}/M_{00}$, $c = \mu_{02}/M_{00}$:

$$\lambda_{1,2}=\frac{(a+c)\pm\sqrt{(a-c)^2+4b^2}}{2}$$

3. Okno o nowym rozmiarze staje się oknem startowym w następnej klatce (stąd „continuously adaptive”).

### Jak to czytać

Momenty rzędu zerowego i pierwszego odpowiadały na pytania „ile?” i „gdzie?”. Momenty rzędu drugiego odpowiadają na pytanie „jaki kształt?”:

- $\mu_{20}/M_{00}$ i $\mu_{02}/M_{00}$ to wariancje położenia w poziomie i w pionie, czyli miara tego, jak szeroko plama jest rozrzucona wokół środka.
- $\mu_{11}/M_{00}$ to kowariancja. Jest różna od zera, gdy plama jest pochylona.
- Wektory własne macierzy $C$ wyznaczają osie plamy, a pierwiastki z wartości własnych to odchylenia standardowe wzdłuż tych osi. Jest to ta sama operacja co analiza głównych składowych (PCA) na chmurze punktów.
- Odchylenie standardowe nie jest jeszcze długością, stąd współczynnik $k$. W OpenCV $k = 4$, bo dla równomiernie wypełnionej elipsy pełna długość osi wynosi dokładnie 4 odchylenia standardowe.

Dzielenie przez $M_{00}$ jest konieczne: bez niego rozmiar okna zależałby od jasności mapy $P$, a nie tylko od kształtu plamy.

### Przykład

Trzy piksele o wartości $P = 1$ w punktach $(0,0), (1,2), (2,4)$, czyli na prostej $y = 2x$.

- $M_{00} = 3$, środek masy $(1, 2)$
- $\mu_{20} = 1 + 0 + 1 = 2$, $\mu_{02} = 4 + 0 + 4 = 8$, $\mu_{11} = 2 + 0 + 2 = 4$
- $\theta = \frac{1}{2}\mathrm{atan2}(8,\ 2 - 8) = 63{,}43°$, czyli dokładnie nachylenie prostej ($\arctan 2$)
- wartości własne $C$: $\lambda_1 = 3{,}33$, $\lambda_2 = 0$. Cały rozrzut leży wzdłuż jednej osi, a szerokość jest zerowa, bo punkty leżą na prostej.

Zwykły jednoargumentowy arcus tangens dałby tutaj $\frac{1}{2}\arctan\frac{8}{-6} = -26{,}57°$, czyli kierunek prostopadły do prostej. Dlatego we wzorze jest atan2.

### CamShift w OpenCV

- Okno przeszukiwania jest zawsze prostokątem o bokach równoległych do osi obrazu.
- Funkcja `cv.CamShift` zwraca dwie rzeczy: obrócony prostokąt (środek, rozmiar, kąt), który jest wynikiem śledzenia i służy do narysowania, oraz nieobrócone okno, z którego należy wystartować w następnej klatce.
- W zwracanym prostokącie pierwszy wymiar to bok krótszy, a kąt jest równy $\theta + 90°$ (z dokładnością do 180°, bo prostokąt obrócony o 180° wygląda tak samo).
- Ponieważ rozmiar okna z jednej klatki jest punktem wyjścia w następnej, błąd może narastać: jeśli tło tuż przy obiekcie ma podobny kolor, okno może rosnąć z klatki na klatkę, aż obejmie cały pasujący obszar.

Przykładowa implementacja w Pythonie [camshift](src/camshift.py). (Uwaga: w NumPy 2.x nie ma już funkcji `pts = np.int0(pts)` trzeba ją zamienić na `pts = pts.astype(np.int32)`.)

## Optical flow (przepływ optyczny)

Przepływ optyczny to wzór pozornego ruchu obiektów na obrazie między dwiema kolejnymi klatkami, spowodowany ruchem obiektu lub kamery. Reprezentowany jest przez dwuwymiarowe pole wektorowe, w którym każdy wektor jest wektorem przemieszczenia pokazującym ruch punktów z pierwszej klatki do drugiej.

Przepływ optyczny opiera się na kilku założeniach:

- Intensywność pikseli obiektu nie zmienia się między kolejnymi klatkami.
- Sąsiednie piksele charakteryzują się podobnym ruchem.

Rozważmy piksel $I(x,y,t)$ w pierwszej klatce (zwróć uwagę, że dodano tu nowy wymiar – czas).
Przesuwa się on o odległość $(dx, dy)$ w następnej klatce zarejestrowanej po upływie czasu $dt$. Ponieważ piksele te są identyczne, a ich jasność nie ulega zmianie, możemy stwierdzić, że:

$$I(x,y,t)=I(x+dx,y+dy,t+dt)$$

Następnie należy zastosować przybliżenie szeregiem Taylora pierwszego rzędu wokół punktu $(x,y,t)$:

$$I(x+dx,y+dy,t+dt) \approx I(x,y,t) + \frac{\partial I}{\partial x}dx + \frac{\partial I}{\partial y}dy + \frac{\partial I}{\partial t}dt.$$

Stosując skrócone oznaczenia

$$f_x=\frac{\partial I}{\partial x}, \qquad f_y=\frac{\partial I}{\partial y}, \qquad f_t=\frac{\partial I}{\partial t},$$

otrzymujemy

$$I(x+dx,y+dy,t+dt) \approx I(x,y,t)+f_x\,dx+f_y\,dy+f_t\,dt$$

Podstawiając do równania zachowania jasności:

$$I(x,y,t) = I(x,y,t)+f_x\,dx+f_y\,dy+f_t\,dt$$

a następnie odejmując od obu stron wyrażenie $I(x,y,t)$, dostajemy:

$$0=f_x\,dx+f_y\,dy+f_t\,dt$$

Finalnie, po podzieleniu przez $dt$:

$$0= f_x \frac{dx}{dt} + f_y\frac{dy}{dt} + f_t = f_x u + f_y v + f_t$$

gdzie:

$$u=\frac{dx}{dt};v=\frac{dy}{dt}$$

Powyższe równanie nazywa się równaniem przepływu optycznego. Wielkości $f_{x}$ i $f_{y}$ to gradienty obrazu (jak szybko jasność zmienia się w poziomie i w pionie), które możemy obliczyć z jednej klatki. Analogicznie, $f_t$ jest gradientem w funkcji czasu, czyli różnicą jasności tego samego piksela między klatkami. Jeśli za jednostkę czasu przyjąć jedną klatkę, $(u,v)$ to przesunięcie w pikselach na klatkę.

**Przykład w jednym wymiarze.** Wzdłuż linii obrazu jasność rośnie o 2 na piksel ($f_x = 2$). Między klatkami jasność w ustalonym pikselu spadła o 4 ($f_t = -4$). Z równania $f_x u + f_t = 0$ dostajemy $u = -f_t / f_x = 2$: obraz przesunął się o 2 piksele w prawo. Przesunięcie to „zmiana jasności podzielona przez nachylenie”.

W dwóch wymiarach niewiadome są dwie, $(u,v)$, a równanie mamy tylko jedno, więc potrzebny jest *sprytny* sposób, aby je wyznaczyć.

### Metoda Lucasa-Kanade'a do rozwiązania optical flow

Wcześniej spotkaliśmy się z założeniem, że wszystkie sąsiednie piksele będą wykazywały podobny ruch. Metoda Lucasa-Kanade'a rozpatruje małe okno wokół danego punktu i przyjmuje, że wszystkie jego piksele poruszają się tak samo. W najprostszym wariancie jest to obszar 3x3, czyli 9 punktów. Dla każdego z nich możemy obliczyć $(f_x, f_y, f_t)$, więc zadanie sprowadza się do rozwiązania 9 równań z dwiema niewiadomymi, co stanowi układ nadokreślony. Rozwiązuje się go metodą najmniejszych kwadratów. Prowadzi ona do poniższego układu dwóch równań z dwiema niewiadomymi:

$$\begin{bmatrix} u\\ v \end{bmatrix} = \begin{bmatrix} \sum_i f_{x_i}^{2} & \sum_i f_{x_i}f_{y_i}\\ \sum_i f_{x_i}f_{y_i} & \sum_i f_{y_i}^{2} \end{bmatrix}^{-1} \begin{bmatrix} -\sum_i f_{x_i}f_{t_i}\\ -\sum_i f_{y_i}f_{t_i} \end{bmatrix}$$

W przykładowym skrypcie okno ma rozmiar 15x15 (parametr `winSize`), czyli równań jest 225; wzór pozostaje ten sam.

**Dlaczego śledzi się rogi.** Jednoznaczne rozwiązanie istnieje tylko wtedy, gdy macierz w powyższym wzorze da się odwrócić, a to wymaga, żeby w oknie występowały gradienty o różnych kierunkach. Przykład: okno 3x3, w którym jasność zmienia się tylko w poziomie (np. pionowa krawędź), czyli w każdym z 9 pikseli $f_x = 1$, $f_y = 0$. Macierz ma wtedy postać

$$\begin{bmatrix} 9 & 0\\ 0 & 0 \end{bmatrix}$$

i wyznacznik 0, więc nie da się jej odwrócić. Zgadza się to z intuicją: patrząc przez małe okno na prostą krawędź, nie da się stwierdzić, czy przesuwa się ona także wzdłuż samej siebie. Na jednolitej powierzchni jest jeszcze gorzej (cała macierz jest zerowa). Dlatego do śledzenia wybiera się rogi; w skrypcie robi to funkcja `cv.goodFeaturesToTrack`.

**Duże ruchy i piramida.** Jeżeli na obrazie pojawiają się stosunkowo duże ruchy, metoda może nie zadziałać, ponieważ przybliżenie pierwszego rzędu jest dobre tylko dla małych przesunięć. Aby temu zaradzić, stosuje się wieloskalowe przetwarzanie obrazu, czyli tzw. piramidę: kolejne poziomy to ten sam obraz w coraz mniejszej rozdzielczości. Duży ruch na pomniejszonym obrazie staje się małym ruchem, więc przepływ wyznacza się najpierw na najmniejszym obrazie, a potem poprawia na kolejnych, coraz większych. W skrypcie `maxLevel=2` oznacza trzy poziomy; na najwyższym obraz jest pomniejszony 4 razy, więc ruch o 8 pikseli to tam 2 piksele.

**Uwaga na `status`.** Funkcja `cv.calcOpticalFlowPyrLK` zwraca dla każdego punktu `status`. Wartość 1 oznacza tylko, że algorytm znalazł jakieś dopasowanie, a nie że jest ono poprawne. Gdy śledzony punkt zniknie z obrazu, funkcja może wskazać inny punkt, który wygląda podobnie.

Przykładowa implementacja w Pythonie [Lucas Kanade optical flow](src/Lucas-Kanade_optical_flow.py)

### Dense Optical Flow (gęsty przepływ optyczny)

Metoda Lucasa-Kanade'a oblicza przepływ optyczny dla rzadkiego zbioru cech (w naszym przykładzie są to rogi wykryte za pomocą odpowiednich detektorów). OpenCV udostępnia inny algorytm służący do obliczania gęstego przepływu optycznego, czyli wyznaczonego dla każdego punktu obrazu. Opiera się on na algorytmie Gunnara Farnebäcka, który został opisany w
[Two-Frame Motion Estimation Based on Polynomial Expansion](https://link.springer.com/chapter/10.1007/3-540-45103-x_50).
W tym podejściu zamiast liczyć pochodne obrazu i rozwiązywać równania przepływu optycznego, aproksymuje się lokalne fragmenty obrazu wielomianami drugiego stopnia (paraboloidą).
Dopasowania poszczególnych fragmentów obrazu (czyli optical flow) znajduje się poprzez lokalne dopasowanie aproksymacji powierzchni na dwóch porównywanych ze sobą obrazach.

Dodatkowo dokonywana jest też korekcja szumów: dla każdego piksela wyznacza się rozwiązanie w pewnym sąsiedztwie i znajduje się rozwiązanie optymalne, sprowadzając problem do metody najmniejszych kwadratów.

Dla dużych przemieszczeń stosowana jest piramida, gdzie kolejne poziomy mają coraz mniejszą rozdzielczość.

Wynikiem jest tablica o rozmiarze obrazu z dwiema liczbami $(u,v)$ dla każdego piksela. Dwa przykładowe skrypty różnią się tylko sposobem jej pokazania: pierwszy koduje kierunek ruchu barwą, a szybkość jasnością (względem najszybszego punktu w danej klatce), drugi rysuje strzałki w węzłach siatki.

Przykładowa implementacja w Pythonie [dense optical flow - wizualizacja 1](src/Dense_optical_flow.py)

Przykładowa implementacja w Pythonie [dense optical flow - wizualizacja 2](src/Dense_optical_flow_vectors.py)

## Filtr Kalmana

Filtr Kalmana (Kalman Filter) to algorytm służący do estymacji stanu dynamicznego układu na podstawie zaszumionych pomiarów. Algorytm ten pozwala wyznaczyć najlepsze przybliżenie położenia, prędkości lub innych parametrów obiektu, nawet gdy pomiary są niedokładne.
Zakładamy, że zjawisko da się opisać modelem liniowym, a zarówno pomiar, jak i proces przetwarzania wewnątrz układu (sam model ruchu) są obarczone błędem o rozkładzie gaussowskim.

Idea: mamy dwa niedoskonałe źródła informacji o położeniu obiektu. Pierwszym jest model ruchu („obiekt porusza się mniej więcej ze stałą prędkością, więc powinien być tutaj”), drugim pomiar („detektor mówi, że jest tam”). Filtr w każdym kroku łączy je w średnią ważoną, a wagi dobiera według tego, któremu źródłu można w danej chwili bardziej ufać.

### Model

Oznaczenia: pogrubione litery ($\mathbf{x}_k$, $\mathbf{z}_k$) to wektory, zwykłe ($x$, $y$, $v_x$, $v_y$) to pojedyncze liczby. Indeks $k$ numeruje kolejne chwile (klatki). Litery $P$, $H$ i $k$ mają w tej sekcji inne znaczenie niż w sekcjach MeanShift i CamShift (tam: mapa prawdopodobieństwa, histogram, numer iteracji lub współczynnik skali); zachowano je, bo takie oznaczenia są przyjęte w literaturze o filtrze Kalmana i w OpenCV.

Załóżmy, że chcemy zamodelować ruch 1D lub 2D. Stan układu opisuje wektor:

1D

$\mathbf{x}_k=[x,v_x]^{T}$

2D

$\mathbf{x}_k=[x,y,v_x,v_y]^{T}$

Gdzie: $x,y$ - współrzędne położenia, $v_x,v_y$ - składowe prędkości.

Równanie ruchu ma postać:

$$\mathbf{x}_k= F \mathbf{x}_{k-1} + \mathbf{w}_k$$

Gdzie $F$ jest macierzą przejścia stanu, $\mathbf{w}_k$ jest szumem procesu (wszystkim tym, czego model ruchu nie uwzględnia, np. hamowaniem lub skrętem).

Dla ruchu jednostajnego:

dla 1D:

$$F= \begin{bmatrix} 1&\Delta t\\0&1\end{bmatrix},$$

dla 2D

$$F= \begin{bmatrix} 1&0&\Delta t&0\\ 0&1&0&\Delta t\\ 0&0&1&0\\ 0&0&0&1 \end{bmatrix},$$

co po rozpisaniu na współrzędne oznacza: nowe położenie to stare położenie plus prędkość razy czas, a prędkości pozostają bez zmian:

$$x \leftarrow x+v_x\Delta t,$$

$$y \leftarrow y+v_y\Delta t.$$

Pomiar opisuje równanie:

$$\mathbf{z}_k= H \mathbf{x}_k + \mathbf{v}_k$$

!!!!Uwaga!!!! Wartością mierzalną (czy też obserwowalną przez nas) jest $\mathbf{z}_k$, a nie $\mathbf{x}_k$!

dla 1D

$$H= [1,0]$$

dla 2D

$$H= \begin{bmatrix} 1&0&0&0\\ 0&1&0&0 \end{bmatrix},$$

Gdzie: $H$ - macierz obserwacji (mierzymy tylko pozycje, prędkości nie), $\mathbf{v}_k$ - szum pomiarowy (wektor; nie mylić ze składowymi prędkości $v_x, v_y$).

Dodatkowo musimy zainicjować macierz $P$, czyli macierz kowariancji błędu estymaty stanu (niepewność naszej wiedzy o stanie). Jest to macierz kwadratowa o boku równym długości wektora stanu. Na początek przyjmuje się zwykle macierz diagonalną, np.:

dla 1D:

$$P = \begin{bmatrix} 100 & 0\\ 0 & 100 \end{bmatrix},$$

dla 2D:

$$P = \begin{bmatrix} 100&0&0&0\\ 0&100&0&0\\ 0&0&100&0\\ 0&0&0&100 \end{bmatrix},$$

Gdzie im większe liczby na diagonali, tym większa niepewność oszacowania, np. 100 - duża niepewność, 1 - mała niepewność.

Macierz kowariancji szumu procesu (macierz $Q$ opisuje szum procesu, czyli niepewność modelu ruchu) ma ten sam wymiar co $P$:

dla 1D:

$$Q = \begin{bmatrix} 0.01 & 0\\ 0 & 0.01 \end{bmatrix},$$

dla 2D:

$$Q = \begin{bmatrix} 0.01&0&0&0\\ 0&0.01&0&0\\ 0&0&0.01&0\\ 0&0&0&0.01 \end{bmatrix},$$

Kowariancja szumu pomiarowego $R$ (macierz kwadratowa o boku równym długości wektora pomiaru):

1D

$$R=\begin{bmatrix} \sigma_{x}^2\end{bmatrix}$$

2D:
$$R=\begin{bmatrix} \sigma_{x}^2 & 0\\ 0 & \sigma_{y}^2 \end{bmatrix}$$

Gdzie $\sigma_{x}^2, \sigma_{y}^2$ - wariancje błędu pomiaru ($\sigma_x, \sigma_y$ to odchylenia standardowe). Przykład: jeśli pomiar położenia myli się typowo o 2 piksele ($\sigma = 2$), to w macierzy $R$ wpisujemy 4.

Zestawienie dla ruchu 2D (stan ma 4 składowe, pomiar 2):

| Symbol | Znaczenie | Wymiar | Skąd się bierze |
|---|---|---|---|
| $\mathbf{x}_k$ | stan: położenie i prędkość | 4 × 1 | szacuje go filtr |
| $\mathbf{z}_k$ | pomiar: położenie | 2 × 1 | z detektora lub czujnika |
| $F$ | model ruchu | 4 × 4 | ustalamy sami |
| $H$ | co z wektora stanu mierzymy | 2 × 4 | ustalamy sami |
| $Q$ | jak bardzo model ruchu może się mylić | 4 × 4 | dobieramy |
| $R$ | jak bardzo pomiar może się mylić | 2 × 2 | z własności czujnika lub dobieramy |
| $P$ | jak niepewna jest bieżąca estymata stanu | 4 × 4 | podajemy wartość początkową, dalej liczy ją filtr |
| $K$ | wzmocnienie Kalmana: waga pomiaru | 4 × 2 | liczy je filtr |

### Algorytm filtracji Kalmana działa w następujący sposób:

W każdym kroku istnieją dwie wersje estymaty stanu. Oznaczamy je: $\hat{\mathbf{x}}_k^-$ i $P_k^-$ to estymata stanu i jej niepewność **przed** uwzględnieniem pomiaru z chwili $k$ (wynik predykcji), a $\hat{\mathbf{x}}_k$ i $P_k$ to te same wielkości **po** korekcji pomiarem. Daszek odróżnia estymatę od prawdziwego stanu $\mathbf{x}_k$, którego nie znamy.

**Predykcja**

Krok 1. Załóżmy, że mamy estymatę stanu z poprzedniego kroku $\hat{\mathbf{x}}_{k-1}$. Pierwszym krokiem filtracji Kalmana jest predykcja stanu:

$$\hat{\mathbf{x}}_{k}^- = F \hat{\mathbf{x}}_{k-1}$$

Krok 2. Drugim krokiem predykcji jest aktualizacja macierzy kowariancji błędu:

$$P_k^- = F P_{k-1} F^{T} + Q$$

W naszym przykładzie predykcja zwiększa niepewność położenia (niepewność prędkości przenosi się na położenie, a do tego dochodzi $Q$): im dłużej tylko przewidujemy, tym mniej wiemy.

**Korekcja**

Krok 3. Po otrzymaniu nowego pomiaru obliczana jest innowacja (jest to różnica między pomiarem a przewidywaniem):

$$\tilde{\mathbf{y}}_k = \mathbf{z}_k - H \hat{\mathbf{x}}_k^-$$

Krok 4. Następnie wyznacza się kowariancję innowacji:

$$S_k = H P_k^- H^T + R$$

Krok 5. Obliczamy wzmocnienie Kalmana:

$$K_k = P_k^- H^{T} S_k^{-1}$$

Macierz $K_k$ określa, jak bardzo należy ufać pomiarowi w stosunku do modelu ruchu. Małe $R$ powoduje wzrost $K$, czyli większe zaufanie do pomiaru. Duże $R$ zmniejsza $K$, czyli większe zaufanie do modelu.

Krok 6. Kolejnym krokiem jest aktualizacja stanu. Nowy stan wyznaczany jest ze wzoru:

$$\hat{\mathbf{x}}_k = \hat{\mathbf{x}}_k^- + K_k \tilde{\mathbf{y}}_k$$

Krok 7. Ostatnim etapem jest aktualizacja niepewności. Niepewność po korekcji opisywana jest wzorem:

$$P_k = (I - K_k H) P_k^-$$

Gdzie $I$ to macierz jednostkowa. Po uwzględnieniu nowego pomiaru niepewność estymacji maleje (a w najgorszym razie się nie zmienia).

Kroki 1–7 powtarza się dla każdej kolejnej chwili $k$.

### Jak czytać wzmocnienie $K$

W najprostszym przypadku, gdy stan jest jedną liczbą i mierzymy go bezpośrednio ($H = 1$), wzory upraszczają się do:

$$K = \frac{P^-}{P^- + R}, \qquad \hat{x} = \hat{x}^- + K\,(z - \hat{x}^-) = (1-K)\,\hat{x}^- + K\,z$$

Wynik jest więc średnią ważoną predykcji i pomiaru, a $K$ (liczba od 0 do 1) jest wagą pomiaru.

- Gdy predykcja jest bardzo niepewna ($P^-$ duże względem $R$), $K$ jest bliskie 1 i filtr przyjmuje prawie sam pomiar.
- Gdy pomiar jest bardzo zaszumiony ($R$ duże względem $P^-$), $K$ jest bliskie 0 i filtr trzyma się predykcji.

### Przykład: jeden krok filtru 1D

Parametry jak w skrypcie `kalman_1d.py`: $\Delta t = 1$, $Q = 0{,}01 \cdot I$, $R = [4]$, stan początkowy $\hat{\mathbf{x}}_0 = [0;\ 1]^T$ (położenie 0, prędkość 1), $P_0 = 10 \cdot I$. Przychodzi pomiar $z_1 = 2{,}5$.

| Krok | Obliczenie | Wynik |
|---|---|---|
| predykcja stanu | $F \hat{\mathbf{x}}_0$ | $\hat{\mathbf{x}}_1^- = [1;\ 1]^T$ |
| predykcja niepewności | $F P_0 F^T + Q$ | $P_1^-$: na przekątnej $20{,}01$ i $10{,}01$, poza przekątną $10$ |
| innowacja | $2{,}5 - 1$ | $1{,}5$ |
| kowariancja innowacji | $20{,}01 + 4$ | $S_1 = 24{,}01$ |
| wzmocnienie | $[20{,}01;\ 10]^T / 24{,}01$ | $K_1 = [0{,}833;\ 0{,}416]^T$ |
| korekcja stanu | $[1;\ 1]^T + K_1 \cdot 1{,}5$ | $\hat{\mathbf{x}}_1 = [2{,}25;\ 1{,}62]^T$ |
| korekcja niepewności | $(I - K_1 H) P_1^-$ | przekątna $P_1$: $3{,}33;\ 5{,}85$ |

Co widać w tym przykładzie:

- Estymata położenia (2,25) leży między predykcją (1) a pomiarem (2,5), bliżej pomiaru, bo na początku niepewność stanu jest duża w porównaniu z $R$.
- Filtr poprawił także prędkość (z 1 na 1,62), chociaż prędkości nikt nie mierzy. Dzieje się tak dzięki elementowi poza przekątną macierzy $P_1^-$ (równemu 10), który wiąże błąd położenia z błędem prędkości.
- Niepewność położenia spadła z 20,01 do 3,33.

### Filtr Kalmana w OpenCV

W klasie `cv2.KalmanFilter` wielkościom ze wzorów odpowiadają pola:

| Wzór | Pole | Uwagi |
|---|---|---|
| $\hat{\mathbf{x}}_k^-$ | `statePre` | zwraca je `predict()` |
| $\hat{\mathbf{x}}_k$ | `statePost` | zwraca je `correct(z)`; tu wpisuje się stan początkowy |
| $P_k^-$ | `errorCovPre` | |
| $P_k$ | `errorCovPost` | tu wpisuje się niepewność początkową |
| $F$ | `transitionMatrix` | w dokumentacji OpenCV oznaczana literą $A$ |
| $H$ | `measurementMatrix` | |
| $Q$ | `processNoiseCov` | |
| $R$ | `measurementNoiseCov` | |
| $K_k$ | `gain` | |

Przykładowa implementacja w Pythonie [filtru Kalmana 1D](src/kalman_1d.py)

Przykładowa implementacja w Pythonie [filtru Kalmana 2D](src/kalman_2d.py). Uwaga: zielona krzywa podpisana na wykresach „Kalman” („Kalman X”, „Kalman Y”) to wynik `kf.predict()`, czyli predykcja $\hat{\mathbf{x}}_k^-$ sprzed uwzględnienia pomiaru. Estymatę po korekcji $\hat{\mathbf{x}}_k$ zwraca `kf.correct()`.

## Literatura

[Zhu, Li, and Tao Hu. "Research of CamShift Algorithm to Track Motion Objects." TELKOMNIKA Indonesian Journal of Electrical Engineering 11.8 (2013): 4372-4378.](https://journal.esperg.com/index.php/TIJEE/article/view/2495)

Bradski, G.R. "Real time face and object tracking as a component of a perceptual user interface." Proceedings Fourth IEEE Workshop on Applications of Computer Vision (WACV '98), 1998, pp. 214-219.

[Tutorial OpenCV meanshift, camshift](https://docs.opencv.org/4.13.0/d7/d00/tutorial_meanshift.html)

[Tutorial OpenCV optical flow](https://docs.opencv.org/3.4.20/d4/dee/tutorial_optical_flow.html)

[Dokumentacja OpenCV filtru Kalmana](https://docs.opencv.org/4.7.0/dd/d6a/classcv_1_1KalmanFilter.html)

[Jang, Jennifer, and Heinrich Jiang. "MeanShift++: Extremely fast mode-seeking with applications to segmentation and object tracking." 2021 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR). IEEE, 2021.](https://arxiv.org/pdf/2104.00303)

[Nguyen, Ngoc-Quan, et al. "Real time human tracking using improved CAM-shift." 2017 Joint 17th World Congress of International Fuzzy Systems Association and 9th International Conference on Soft Computing and Intelligent Systems (IFSA-SCIS). IEEE, 2017.](https://ieeexplore.ieee.org/abstract/document/8023295)

[Farnebäck, Gunnar. "Two-frame motion estimation based on polynomial expansion." Scandinavian conference on Image analysis. Berlin, Heidelberg: Springer Berlin Heidelberg, 2003.](https://link.springer.com/chapter/10.1007/3-540-45103-x_50)

[Edward V. Stansfield CMath CSci FIMA, Kalman Filter Tutorial Part I – Theory and Practice](https://cdn.ima.org.uk/wp/wp-content/uploads/2024/09/Kalman-Filter-Tutorial-Part-1_Mathematics-Today_2016.pdf)